# 🧠 Aula 02b: A Equação Normal e a Pseudoinversa na Regressão Linear

Nesta aula complementar, vamos aprofundar na matemática por trás do treinamento de uma Regressão Linear. Veremos:
1. **Revisão do Gradiente Descendente básico:** O ajuste iterativo dos pesos e viés separadamente.
2. **Gradiente Descendente com Vetorização (Bias Embutido):** Como simplificar o código empilhando uma coluna de `1`s (vetor de viés) diretamente na matriz de entrada $X$.
3. **Equação Normal (Solução Analítica):** A fórmula matemática fechada que encontra os melhores parâmetros de uma só vez, sem precisar de loops ou taxa de aprendizado, usando a Pseudoinversa de Moore-Penrose!

<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula02b%20-%20normal%20equation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [89]:
# Carrega o dataset de diabetes e separa as características (X) e os valores reais (y)
from sklearn.datasets import load_diabetes
data = load_diabetes()
X, y = data.data, data.target

In [90]:
# Divide o dataset em 80% para treino e 20% para teste, definindo uma semente aleatória (42) para consistência
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [91]:
# Importa a métrica oficial de Erro Quadrático Médio (MSE) do Scikit-Learn
from sklearn.metrics import mean_squared_error

### 📉 1. Gradiente Descendente Básico (Ajuste Iterativo Separado)

#### 🔍 O que este bloco faz?
Ele implementa uma classe de Regressão Linear que herda da estrutura padrão do Scikit-Learn (`BaseEstimator` e `RegressorMixin`). No método `fit`, ela ajusta os coeficientes (`coefs_`) e o viés/intercepto (`intercept_`) de forma iterativa usando a regra do Gradiente Descendente.

#### 🎯 Qual a intenção pedagógica?
Mostrar como os pesos ($w$) e o intercepto ($b$) são atualizados separadamente a cada iteração:
- Os pesos são atualizados multiplicando a transposta de $X$ pelo vetor de erros.
- O intercepto é atualizado com base na soma simples do erro.
Isso ajuda a fixar a intuição de que o aprendizado é um processo de refinamento contínuo dos parâmetros da reta para minimizar o erro.

In [92]:
from sklearn.base import BaseEstimator, RegressorMixin
import numpy as np

# Regressor Linear básico utilizando Gradiente Descendente iterativo
class LinearRegressor(BaseEstimator, RegressorMixin):
  def __init__(self, max_iter=1000, learning_rate=0.001):
    self.max_iter = max_iter
    self.learning_rate = learning_rate

  def fit(self, X, y):
    # Inicializa os pesos e o intercepto com valores aleatórios
    self.coefs_ = np.random.rand(X.shape[1])
    self.intercept_ = np.random.rand()
    for i in range(self.max_iter):
      # Realiza a predição para o conjunto de dados atual
      y_pred = self.predict(X)
      # Calcula o erro (resíduo)
      error = y - y_pred
      # Atualiza os coeficientes multiplicando X transposta pelo erro e pela taxa de aprendizado
      self.coefs_ += X.T @ error * self.learning_rate
      # Atualiza o intercepto multiplicando a soma dos erros pela taxa de aprendizado
      self.intercept_ += error.sum() * self.learning_rate
    return self

  def predict(self, X):
    # Equação linear padrão: y = Xw + b
    y_pred = X @ self.coefs_ + self.intercept_
    return y_pred.reshape(X.shape[0], )

# Instancia, treina o regressor básico e exibe os pesos aprendidos e o MSE
regressor = LinearRegressor()
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_train)
print(regressor.coefs_, regressor.intercept_)
print(mean_squared_error(y_train, y_pred))

[  57.62561103  -92.09683559  363.63878134  250.97093923   -0.88016353
  -37.22731305 -182.68232011  148.90116566  286.86456729  148.67099638] 151.84790486949117
3142.2472033718404


### 🔌 2. Gradiente Descendente com Vetorização (Bias Embutido)

#### 🔍 O que a função `include_bias` faz?
Ela pega a nossa matriz $X$ e adiciona (empilha horizontalmente) uma coluna cheia do número $1$ à esquerda.

#### 🎯 Qual a intenção pedagógica?
Ao invés de tratar os pesos ($w$) e o intercepto ($b$) como duas variáveis diferentes na equação $y = Xw + b$, nós embutimos o intercepto como o primeiro elemento do vetor de pesos $w$ (vamos chamá-lo de $w_0$).
Para que a multiplicação de matrizes funcione, adicionamos uma coluna de $1$s à esquerda de $X$. Assim:
$$y = w_0 \cdot 1 + w_1 \cdot x_1 + w_2 \cdot x_2 + \dots$$
Isso simplifica drasticamente as operações matemáticas e unifica a atualização dos parâmetros em um único passo vetorizado:
`self.w_ += X.T @ error * self.learning_rate`
Isso ilustra o poder da vetorização matemática em Machine Learning!

In [105]:
from sklearn.base import BaseEstimator, RegressorMixin
import numpy as np

# Função auxiliar para empilhar uma coluna de 1s à esquerda da matriz X
def include_bias(X):
  return np.hstack((np.ones((X.shape[0], 1)), X))

# Regressor Linear otimizado com vetorização e bias embutido
class LinearRegressor(BaseEstimator, RegressorMixin):
  def __init__(self, max_iter=1000, learning_rate=0.005):
    self.max_iter = max_iter
    self.learning_rate = learning_rate

  def fit(self, X, y):
    # Adiciona a coluna de bias à matriz de entrada
    X = include_bias(X)
    # Inicializa os pesos (incluindo o termo de bias w0 na primeira posição)
    self.w_ = np.random.rand(X.shape[1])
    for i in range(self.max_iter):
      # Predição linear vetorizada pura: y = Xw
      y_pred = X @ self.w_
      # Calcula o vetor de erros
      error = y - y_pred
      # Atualização simultânea de todos os pesos em uma única operação matricial
      self.w_ += X.T @ error * self.learning_rate
    return self

  def predict(self, X):
    # Adiciona a coluna de bias e realiza o produto matricial
    X = include_bias(X)
    y_pred = X @ self.w_
    return y_pred.reshape(X.shape[0], )

# Instancia, treina o regressor vetorizado e exibe os pesos vetorizados e o MSE resultante
regressor = LinearRegressor()
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_train)
print(regressor.w_)
print(mean_squared_error(y_train, y_pred))

[ 151.30682882   39.62671817 -234.00881384  546.5442233   339.08734803
  -93.91558466 -128.24658275 -216.97915194  147.74908728  406.57101804
   77.69461265]
2898.8951858961277


### 🧮 3. Equação Normal (Solução Analítica)

#### 🔍 O que este bloco faz?
Ele implementa a **Equação Normal**, que é a solução analítica exata para a Regressão Linear por Mínimos Quadrados. Em vez de realizar passos iterativos, os coeficientes ideais são encontrados diretamente resolvendo a equação normal:
$$w = (X^T X)^{-1} X^T y$$
Como a matriz $X^T X$ pode não ser invertível em alguns casos, o código utiliza a **Pseudoinversa de Moore-Penrose** (`np.linalg.pinv(X)`), que calcula $(X^T X)^{-1} X^T$ de forma muito mais estável.

#### 🎯 Qual a intenção pedagógica?
Demonstrar que, para problemas de regressão linear simples, não precisamos necessariamente de Gradiente Descendente ou loops iterativos. Conseguimos calcular de forma exata e em uma única operação matricial os melhores pesos possíveis.
Compare o MSE obtido aqui com o MSE dos métodos iterativos anteriores. A Equação Normal atinge o mínimo global instantaneamente!

In [100]:
from sklearn.base import BaseEstimator, RegressorMixin
import numpy as np

def include_bias(X):
  return np.hstack((np.ones((X.shape[0], 1)), X))

# Regressor Linear que resolve os mínimos quadrados analiticamente via Pseudoinversa
class LinearRegressor(BaseEstimator, RegressorMixin):
  def fit(self, X, y):
    # Adiciona a coluna de bias
    X = include_bias(X)
    # Resolve diretamente para self.w_ usando a Pseudoinversa de Moore-Penrose
    self.w_ = np.linalg.pinv(X) @ y
    return self

  def predict(self, X):
    X = include_bias(X)
    y_pred = X @ self.w_
    return y_pred.reshape(X.shape[0], )

# Instancia e executa o regressor analítico
regressor = LinearRegressor()
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_train)
print("Pesos analíticos ideais encontrados:", regressor.w_)
print("MSE ótimo no conjunto de treino:", mean_squared_error(y_train, y_pred))

[ 151.34560454   37.90402135 -241.96436231  542.42875852  347.70384391
 -931.48884588  518.06227698  163.41998299  275.31790158  736.1988589
   48.67065743]
2868.5497028355776
